# Deep Learning y Modelos Globales en Series Temporales

## Sesi?n 7: Del M?dico de Pueblo al Hospital de Referencia

Hasta ahora hemos construido modelos que conocen a un ?nico paciente en profundidad. El Random Forest de la sesi?n 5 aprendi? los ritmos de una sola subestaci?n el?ctrica; el modelo de Feature Engineering de la sesi?n 6 extrajo toda la inteligencia posible de una sola serie de ventas. Estos son **Modelos Locales**: especialistas brillantes en un problema concreto.

Pero el mundo real no tiene un paciente. Tiene miles.

Imagina a un m?dico de pueblo que lleva 20 a?os tratando a los mismos vecinos. Conoce perfectamente a cada uno: sus alergias, sus estacionalidades (el se?or Garc?a siempre empeora en invierno), sus reca?das. Si ese paciente es el consumo el?ctrico de una sola subestaci?n, este m?dico es imbatible. Pero el d?a que aparece un paciente nuevo, o que llega una pandemia global (un evento jam?s visto), su experiencia limitada a un solo pueblo se queda corta.

El **Deep Learning** funciona como un gigantesco Hospital de Referencia. En lugar de estudiar a un paciente, analiza los historiales cruzados de cientos de miles de pacientes simult?neamente. Quiz?s no conoce a cada individuo con el mismo detalle inicial, pero ha visto *tantos patrones compartidos* que al ver los primeros s?ntomas (una peque?a ventana de datos), ya sabe reconocer la enfermedad porque la ha visto en otras ciudades, en otros sectores.

A esto se le llama **Cross-Learning**: la red aprende que "un pico de ventas seguido de una ca?da de temperatura" precede a una rotura de stock, sin importar de qu? supermercado o ciudad hablemos. Ha aprendido la f?sica del problema, no solo un ?nico caso.

---

## Objetivos

Al terminar esta sesi?n deber?as ser capaz de:

- **Distinguir** entre Modelos Locales (un modelo por serie) y Modelos Globales (un modelo para todas las series).
- **Entender** por qu? las Redes Neuronales necesitan datos en formato 3D (Tensores) en lugar del formato 2D tabular que usan los ?rboles de decisi?n.
- **Construir** una clase `Dataset` de PyTorch que convierta un DataFrame en ventanas de tiempo listas para entrenar.
- **Comprender** la diferencia conceptual entre un MLP (con amnesia temporal) y una LSTM (con memoria a largo plazo).
- **Implementar** una red LSTM de forecasting usando PyTorch Lightning para evitar el c?digo repetitivo de entrenamiento.
- **Razonar** sobre cu?ndo merece la pena la complejidad del Deep Learning y cu?ndo no.

## Requisitos t?cnicos

Para esta sesi?n necesitamos el entorno `pia-ud1` con las instalaciones correctas. Recuerda que las librer?as de `torch` deben haberse instalado v?a `pip` (no conda-forge) para evitar conflictos con los drivers NVIDIA:

In [ ]:
%%bash
pip install torch pytorch-lightning

Importaciones principales:

In [ ]:
import sys
import os

# os.environ["USE_TF"] = "0"
#os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# os.environ["LIGHTNING_DISABLE_TIPS"] = "1"

sys.path.append("src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from series_temporales import generar_consumo_electrico

---

## 1. El Salto de Paradigma: De 2D a 3D

Para alimentar a un Random Forest (sesi?n 5), us?bamos un DataFrame 2D donde cada fila era un instante de tiempo y cada columna era un `lag` o una variable de calendario. La filosof?a era: *"Dame una fotograf?a del pasado y te dir? qu? ocurrir?"*.

Las Redes Neuronales funcionan de forma distinta. No miran una fotograf?a aislada: miran **una pel?cula completa** (una ventana de tiempo) de un solo vistazo.

Por eso, en lugar de un DataFrame 2D `[Filas, Columnas]`, necesitamos un **Tensor 3D** con la siguiente estructura:

```text
[Batch_Size, Sequence_Length, Features]
```

1. **Batch_Size (El Lote):** Cu?ntas "pel?culas" (ventanas) le mostramos a la red de golpe antes de que actualice sus pesos. Un batch de 32 significa que la red ve 32 ventanas simult?neamente y promedia el aprendizaje de todas ellas. Esto estabiliza el entrenamiento.
2. **Sequence_Length (La Longitud de la Pel?cula):** Cu?ntos pasos hacia atr?s mira la red en cada ventana. Si entrenamos con `sequence_length=24`, la red ve las ?ltimas 24 horas antes de predecir.
3. **Features (Las Variables):** Cu?ntas columnas de informaci?n hay en cada instante. Si usamos consumo, temperatura y d?a de la semana, `features=3`.

*Visualizaci?n:* Imagina un mazo de cartas (el Batch). Cada carta es una peque?a tabla de 24 filas ? 3 columnas, representando lo que ocurri? en las ?ltimas 24 horas.

### Construyendo el Dataset para PyTorch

PyTorch nos obliga a encapsular el mecanismo de extracci?n de ventanas en una clase `Dataset`. Esta clase tiene que saber responder a dos preguntas: ?Cu?ntas ventanas hay? (`__len__`) y ?C?mo me das la ventana n?mero N? (`__getitem__`).

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df, target_col, window_size=24, forecast_horizon=1):
        """
        Convierte un DataFrame temporal en ventanas 3D para PyTorch.

        Args:
            df: DataFrame con ?ndice temporal y las columnas de features + target.
            target_col: Nombre de la columna que queremos predecir.
            window_size: Cu?ntos pasos del pasado mira la red (longitud de la ventana).
            forecast_horizon: Cu?ntos pasos hacia el futuro predecimos.
        """
        self.features = df.drop(columns=[target_col]).values.astype(np.float32)
        self.target = df[target_col].values.astype(np.float32)
        self.window_size = window_size
        self.forecast_horizon = forecast_horizon

        # El n?mero total de ventanas que podemos construir sin salirnos del dataset
        self.length = len(df) - window_size - forecast_horizon + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Ventana de Entrada X: [window_size, n_features]
        X = self.features[idx : idx + self.window_size]

        # Valor objetivo y: el dato que hay 'forecast_horizon' pasos despu?s del final de la ventana
        target_idx = idx + self.window_size + self.forecast_horizon - 1
        y = self.target[target_idx]

        return torch.from_numpy(X), torch.tensor(y)

Usamos el `Dataset` con un `DataLoader` que se encarga de crear los batches autom?ticamente y de mezclar las ventanas de forma segura:

In [ ]:
# Preparamos datos de ejemplo
df = generar_consumo_electrico(
    fecha_inicio="2026-01-01",
    periodos=24*180,
    frecuencia="h",
    incluir_meteorologia=True,
)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").asfreq("h")

# Columnas que usaremos como features
feature_cols = ["consumo_kwh", "temperatura", "hora", "dia_semana"]
df_ml = df[feature_cols].copy().dropna()

# Divisi?n temporal (sin barajar)
n = len(df_ml)
train_df = df_ml.iloc[:int(n * 0.8)]
val_df   = df_ml.iloc[int(n * 0.8):]

scaler = StandardScaler()
train_df = pd.DataFrame(
    scaler.fit_transform(train_df),
    index=train_df.index,
    columns=train_df.columns,
)
val_df = pd.DataFrame(
    scaler.transform(val_df),
    index=val_df.index,
    columns=val_df.columns,
)

train_ds = TimeSeriesDataset(train_df, target_col="consumo_kwh", window_size=24)
val_ds   = TimeSeriesDataset(val_df,   target_col="consumo_kwh", window_size=24)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

# Inspeccionamos un batch
for X_batch, y_batch in train_loader:
    print(f"Shape de X: {X_batch.shape}")   # -> [32, 24, 3] (Batch, Secuencia, Features)
    print(f"Shape de y: {y_batch.shape}")   # -> [32]
    break

> **Pregunta para discutir:** Si la ventana tiene `window_size=24` y `forecast_horizon=1`, ?qu? ocurre matem?ticamente con los ?ltimos 23 datos del conjunto de validaci?n? ?Podemos predecirlos? ?Por qu? aparece `self.length = len(df) - window_size - forecast_horizon + 1` en lugar de simplemente `len(df)`?

---

## 2. El Perceptr?n Multicapa (MLP): El Caso de la Amnesia

La primera red neuronal que estudiaremos es el MLP (*Multi-Layer Perceptron*). Es la versi?n m?s b?sica: un conjunto de capas lineales encadenadas con funciones de activaci?n no lineales (ReLU).

Su problema para las series temporales es la **amnesia temporal**: para poder aceptar el tensor 3D como entrada, primero aplana (`Flatten`) la ventana en un vector 1D largu?simo. Con `window_size=24` y `features=3`, ese vector tiene `24 ? 3 = 72` dimensiones. El MLP las trata exactamente igual: el dato n?mero 1 (hace 24 horas) y el dato n?mero 71 (hace 1 hora) pesan exactamente lo mismo. El orden temporal ha desaparecido.

In [ ]:
class MLPForecaster(pl.LightningModule):
    def __init__(self, window_size, n_features, hidden_dim=64, lr=1e-3):
        super().__init__()
        self.lr = lr
        input_dim = window_size * n_features
        self.net = nn.Sequential(
            nn.Flatten(),                          # Aplana [Batch, Seq, Feat] -> [Batch, Seq*Feat]
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.net(x).squeeze()

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# creamos la red y la entrenamos 

In [ ]:
n_features = len(feature_cols) - 1  # Todas las columnas menos el target

modelo_mlp = MLPForecaster(window_size=24, n_features=n_features)

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",     # Usa GPU si existe, si no, CPU
    log_every_n_steps=10,
    logger=False,
    enable_checkpointing=False,
)

trainer.fit(modelo_mlp, train_loader, val_loader)

In [ ]:
# Predicciones en validaci?n
modelo_mlp.eval()
preds, actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        preds.extend(modelo_mlp(X_batch).numpy())
        actuals.extend(y_batch.numpy())

mae_lstm = mean_absolute_error(actuals, preds)

# Baseline Seasonal Naive (lo que pas? hace 7 d?as = 168 horas)
serie_val = val_df["consumo_kwh"]
serie_escalada = pd.concat([train_df, val_df])["consumo_kwh"]
pred_naive = serie_escalada.shift(168).loc[serie_val.index]
mae_naive = mean_absolute_error(serie_val.dropna(), pred_naive.dropna())

print(f"MAE Seasonal Naive:  {mae_naive:.4f}")
print(f"MAE MLPForecaster:   {mae_lstm:.4f}")

---

## 3. Redes Recurrentes (LSTM): La Cura de la Amnesia

Las LSTMs (*Long Short-Term Memory*) solucionan el problema del MLP con un mecanismo elegante: procesan la ventana temporal **paso a paso**, de principio a fin, manteniendo un **estado oculto** (una memoria interna) que se actualiza con cada nuevo instante.

Es literalmente como leer una novela. Al llegar a la ?ltima p?gina, entiendes la trama no porque recuerdes literalmente cada palabra que le?ste al principio, sino porque tu mente fue construyendo un **contexto acumulado**. La LSTM hace lo mismo con los datos temporales.

La LSTM tiene adem?s dos mecanismos de control internos: una **"compuerta de olvido"** (que decide qu? parte del pasado olvidar) y una **"compuerta de entrada"** (que decide qu? informaci?n nueva a?adir a la memoria). Esta complejidad adicional le cuesta velocidad de entrenamiento, pero le permite capturar dependencias a largo plazo que el MLP no puede ver.

In [ ]:
class LSTMForecaster(pl.LightningModule):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.1, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr

        # La capa LSTM recibe [Batch, Sequence, Features] gracias a batch_first=True
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
        )
        # Proyecci?n final: del ?ltimo estado oculto a 1 valor de predicci?n
        self.head = nn.Linear(hidden_size, 1)
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        # x: [Batch, Sequence, Features]
        lstm_out, _ = self.lstm(x)
        # Tomamos solo la salida del ?ltimo timestep: [Batch, hidden_size]
        last_out = lstm_out[:, -1, :]
        return self.head(last_out).squeeze()

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

### Entrenando con PyTorch Lightning

Lightning nos proporciona el `Trainer`, que gestiona autom?ticamente los bucles de entrenamiento, el movimiento de datos a GPU, y el registro de m?tricas:

In [ ]:
n_features = len(feature_cols) - 1  # Todas las columnas menos el target

modelo = LSTMForecaster(n_features=n_features, hidden_size=64, num_layers=2)

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",     # Usa GPU si existe, si no, CPU
    log_every_n_steps=10,
    logger=False,
    enable_checkpointing=False,
)

trainer.fit(modelo, train_loader, val_loader)

### Evaluando frente al Seasonal Naive

In [ ]:
# Predicciones en validaci?n
modelo.eval()
preds, actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        preds.extend(modelo(X_batch).numpy())
        actuals.extend(y_batch.numpy())

mae_lstm = mean_absolute_error(actuals, preds)

# Baseline Seasonal Naive (lo que pas? hace 7 d?as = 168 horas)
serie_val = val_df["consumo_kwh"]
serie_escalada = pd.concat([train_df, val_df])["consumo_kwh"]
pred_naive = serie_escalada.shift(168).loc[serie_val.index]
mae_naive = mean_absolute_error(serie_val.dropna(), pred_naive.dropna())

print(f"MAE Seasonal Naive:  {mae_naive:.4f}")
print(f"MAE LSTM:            {mae_lstm:.4f}")

> **Pregunta para discutir:** Si el MAE de la LSTM es ligeramente peor que el del Seasonal Naive con solo 180 d?as de datos de entrenamiento, ?es eso una derrota? ?Qu? pasar?a si tuvieras datos de 100 subestaciones distintas? ?C?mo cambiar?a la ecuaci?n?

---

## 4. Actividades de clase

### Actividad 1: La Autopsia del Tensor
Antes de entrenar nada, extrae un ?nico batch del `train_loader` e inspecci?nalo. Imprime su `shape`. Luego extrae el primer elemento del batch (`X_batch[0]`) y compara su contenido con las primeras 24 filas del DataFrame original. ?Coinciden? Esto te confirmar? que has construido correctamente el flujo de ventanas.

### Actividad 2: MLP vs LSTM en Igualdad de Condiciones
Entrena el `MLPForecaster` con exactamente los mismos datos y el mismo n?mero de ?pocas que el `LSTMForecaster`. Compara el `val_loss` final de ambos. ?Cu?l converge m?s r?pido? ?Cu?l tiene un error final menor? ?Observas alg?n signo de *overfitting* (que el `train_loss` baja pero el `val_loss` se estanca o sube)?

### Actividad 3: El Efecto de la Memoria
Reentrena la LSTM cambiando el par?metro `window_size` del Dataset a `12` horas y despu?s a `168` horas (una semana completa). ?C?mo cambia el error de validaci?n? Intenta razonar por qu? una ventana m?s larga no siempre es mejor (pista: m?s ventana = m?s ruido que la LSTM tiene que "memorizar").

---

## 5. Ideas clave

- **Local vs Global:** La gran revoluci?n del Deep Learning en forecasting no est? en la arquitectura, sino en el paradigma. Entrenar con miles de series simult?neamente permite que la red aprenda patrones universales que ning?n modelo cl?sico puede capturar.
- **El Tensor 3D es el lenguaje:** `[Batch, Sequence, Features]` es la gram?tica universal de las redes neuronales para series temporales. Interiorizar esta forma de pensar es m?s importante que memorizar cualquier arquitectura concreta.
- **MLP = Amnesia, LSTM = Memoria:** El MLP trata el tiempo como una tabla desordenada. La LSTM lee el tiempo como una novela, construyendo un contexto acumulado paso a paso.
- **Lightning elimina el ruido:** PyTorch Lightning separa la arquitectura del entrenamiento, permiti?ndote concentrarte en lo que importa: el dise?o de la red.
- **M?s par?metros ? Mejores resultados:** Con 180 d?as de una sola serie, una LSTM puede perder f?cilmente contra el Seasonal Naive. El Deep Learning brilla cuando hay *muchos datos* y *muchas series*.

## Continuaci?n

Ahora que entendemos las LSTMs y sus limitaciones, en la pr?xima sesi?n exploraremos las **Redes Convolucionales Temporales (TCNs)**: una arquitectura alternativa que resuelve el problema de la lentitud de las LSTMs usando convoluciones 1D con un truco matem?tico llamado *dilataci?n*. Las TCNs son paralelizables, m?s r?pidas de entrenar y tienen un campo receptivo que podemos controlar con precisi?n matem?tica.

In [ ]:
#actividad 1
# obtener un batch del train loader
X_batch, y_batch = next(iter(train_loader))
X_batch.shape

In [ ]:
X_batch[0]